# Experiment 3.2: Manifold Structure with Zipf Distribution

This notebook tests whether nonlinear encoders develop manifold structure when trained
on distributions with Zipf (power-law) feature probabilities: p_i ∝ 1/(i+1).

The Zipf distribution creates a realistic hierarchy where some features are much more
common than others, potentially encouraging manifold representations.

**Key questions:**
1. Does manifold structure emerge in MLPAutoencoder with Zipf distributions?
2. How does angular variance compare to uniform sparsity baselines?
3. Do rare vs common features have different manifold properties?

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch import Generator

from occhio import ToyModel, MLPAutoencoder
from occhio.autoencoder import TiedLinear, TiedLinearRelu, TiedMLPEncoder
from occhio.distributions.sparse import SparseUniform
from occhio.analysis import (
    compute_feature_jacobians,
    angular_variance,
    jacobian_pca,
    direction_vs_context,
)

## 1. Configuration

Using plan.md specifications:
- n=200 features, m=20 hidden (10:1 compression)
- Zipf probabilities: p_i = 1/(i+1)

In [ ]:
# Configuration from plan.md section 3.1
N_FEATURES = 200
N_HIDDEN = 20  # 10:1 compression ratio
N_EPOCHS = 25000
BATCH_SIZE = 512
N_JACOBIAN_SAMPLES = 1000

# Zipf distribution: p_i = 1/(i+1)
ZIPF_PROBS = [1 / (i + 1) for i in range(N_FEATURES)]

# For comparison: uniform sparse with same expected density
MEAN_PROB = np.mean(ZIPF_PROBS)

print(f"Configuration: {N_FEATURES} features -> {N_HIDDEN} hidden")
print(f"Compression ratio: {N_FEATURES / N_HIDDEN:.1f}:1")
print(f"Zipf p_active range: [{min(ZIPF_PROBS):.4f}, {max(ZIPF_PROBS):.4f}]")
print(f"Mean p_active: {MEAN_PROB:.4f}")
print(f"Expected features per sample: {sum(ZIPF_PROBS):.1f}")

In [ ]:
# Visualize Zipf distribution
fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(N_FEATURES)), y=ZIPF_PROBS, name="Zipf p_active"))
fig.add_hline(y=MEAN_PROB, line_dash="dash", annotation_text=f"Mean: {MEAN_PROB:.4f}")
fig.update_layout(
    title="Zipf Feature Probabilities: p_i = 1/(i+1)",
    xaxis_title="Feature Index",
    yaxis_title="p_active",
    height=400,
)
fig.show()

## 2. Train Models

Training three architectures:
1. **TiedLinear**: Linear baseline (angular variance should be ~0)
2. **TiedLinearRelu**: Piecewise linear (ReLU decoder)
3. **MLPAutoencoder**: Smooth nonlinearity (GELU encoder)

In [ ]:
def create_zipf_distribution(seed=42):
    """Create SparseUniform with Zipf probabilities."""
    return SparseUniform(
        N_FEATURES,
        [1 / (i + 2) ** 0.5 for i in range(N_FEATURES)],
        generator=Generator().manual_seed(seed),
    )


def create_uniform_distribution(seed=42):
    """Create SparseUniform with uniform probabilities (for comparison)."""
    return SparseUniform(
        n_features=N_FEATURES,
        p_active=MEAN_PROB,
        generator=Generator().manual_seed(seed),
    )


# Verify distribution samples
dist = create_zipf_distribution()
samples = dist.sample(1000)
print(f"Sample shape: {samples.shape}")
print(f"Feature 0 (p=1.00) active rate: {(samples[:, 0] > 0).float().mean():.3f}")
print(f"Feature 50 (p=0.02) active rate: {(samples[:, 50] > 0).float().mean():.3f}")
print(f"Feature 199 (p=0.005) active rate: {(samples[:, 199] > 0).float().mean():.3f}")

In [ ]:
# # Train TiedLinear (linear baseline)
# print("Training TiedLinear (linear encoder)...")
# linear_model = ToyModel(
#     distribution=create_zipf_distribution(),
#     ae=TiedLinear(n_features=N_FEATURES, n_hidden=N_HIDDEN),
# )
# linear_losses, _ = linear_model.fit(
#     n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
# )
# print(f"Final loss: {linear_losses[-1]:.6f}")

In [ ]:
# Train TiedLinearRelu (piecewise linear)
print("Training TiedLinearRelu (ReLU decoder)...")
relu_model = ToyModel(
    distribution=create_zipf_distribution(),
    ae=TiedLinearRelu(n_features=N_FEATURES, n_hidden=N_HIDDEN),
)
relu_losses, _ = relu_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {relu_losses[-1]:.6f}")

In [ ]:
# Train MLPAutoencoder (smooth nonlinearity)
print("Training MLPAutoencoder (RELU encoder)...")
mlp_model = ToyModel(
    distribution=create_zipf_distribution(),
    ae=TiedMLPEncoder(
        [N_FEATURES, N_HIDDEN * 4, N_HIDDEN],
    ),
)
mlp_losses, _ = mlp_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_losses[-1]:.6f}")

In [ ]:
# # Train MLP on uniform distribution for comparison
# print("Training MLPAutoencoder on UNIFORM distribution (comparison)...")
# mlp_uniform_model = ToyModel(
#     distribution=create_uniform_distribution(),
#     ae=MLPAutoencoder(
#         n_features=N_FEATURES,
#         n_hidden=N_HIDDEN,
#         encoder_hidden_dim=N_HIDDEN * 2,
#         activation="relu",
#         decoder_activation="relu",
#     ),
# )
# mlp_uniform_losses, _ = mlp_uniform_model.fit(
#     n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
# )
# print(f"Final loss: {mlp_uniform_losses[-1]:.6f}")

In [ ]:
# Plot training losses
fig = go.Figure()
# fig.add_trace(go.Scatter(y=linear_losses, name="TiedLinear (Zipf)", mode="lines"))
fig.add_trace(go.Scatter(y=relu_losses, name="TiedLinearRelu (Zipf)", mode="lines"))
fig.add_trace(go.Scatter(y=mlp_losses, name="MLP (Zipf)", mode="lines"))
# fig.add_trace(go.Scatter(y=mlp_uniform_losses, name="MLP (Uniform)", mode="lines", line=dict(dash="dash")))
fig.update_layout(
    title="Training Loss Comparison",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    yaxis_type="log",
    height=400,
)
fig.show()

## 3. Experiment A: Does Manifold Structure Emerge?

Compute angular variance for all features across models.
If MLP develops significantly higher AV than linear baseline, manifold structure is present.

In [ ]:
# Generate test samples for Jacobian analysis
test_dist = create_zipf_distribution(seed=999)  # Different seed for test
test_samples = test_dist.sample(N_JACOBIAN_SAMPLES * 2)

# For uniform comparison
test_dist_uniform = create_uniform_distribution(seed=999)
test_samples_uniform = test_dist_uniform.sample(N_JACOBIAN_SAMPLES * 2)

print(f"Test samples shape: {test_samples.shape}")

In [ ]:
def compute_all_angular_variances(model, samples, n_samples_per_feature=500):
    """Compute angular variance for all features."""
    n_features = model.n_features
    avs = []

    for feat_idx in range(n_features):
        # Filter for samples where this feature is active
        active_mask = samples[:, feat_idx] > 0
        active_samples = samples[active_mask]

        if len(active_samples) < 10:
            avs.append(np.nan)  # Not enough samples
            continue

        # Limit to n_samples_per_feature
        active_samples = active_samples[:n_samples_per_feature]

        # Compute Jacobians and angular variance
        jacs = compute_feature_jacobians(model, feat_idx, active_samples)
        av = angular_variance(jacs)
        avs.append(av)

        if feat_idx % 50 == 0:
            print(f"  Feature {feat_idx}: AV = {av:.6f} (n={len(active_samples)})")

    return np.array(avs)


print("Computing angular variance for all features...")
print("\nTiedLinear (Zipf):")
# linear_avs = compute_all_angular_variances(linear_model, test_samples)

print("\nTiedLinearRelu (Zipf):")
relu_avs = compute_all_angular_variances(relu_model, test_samples)

print("\nMLPAutoencoder (Zipf):")
mlp_avs = compute_all_angular_variances(mlp_model, test_samples)

print("\nMLPAutoencoder (Uniform):")
# mlp_uniform_avs = compute_all_angular_variances(mlp_uniform_model, test_samples_uniform)

In [ ]:
# Summary statistics (excluding NaN)
def summarize_avs(avs, name):
    valid = avs[~np.isnan(avs)]
    print(f"{name}:")
    print(f"  Mean AV: {valid.mean():.6f}")
    print(f"  Max AV:  {valid.max():.6f}")
    print(f"  Std AV:  {valid.std():.6f}")
    print(f"  Valid features: {len(valid)}/{len(avs)}")
    return valid


print("=" * 60)
print("ANGULAR VARIANCE SUMMARY")
print("=" * 60)
# linear_valid = summarize_avs(linear_avs, "TiedLinear (Zipf)")
print()
relu_valid = summarize_avs(relu_avs, "TiedLinearRelu (Zipf)")
print()
mlp_valid = summarize_avs(mlp_avs, "MLPAutoencoder (Zipf)")
print()
# mlp_uniform_valid = summarize_avs(mlp_uniform_avs, "MLPAutoencoder (Uniform)")

In [ ]:
# Compute feature reconstruction quality for coloring
# MSE reconstruction loss: 0 = perfect reconstruction
one_hot_reconstructions = []
for feat_idx in range(N_FEATURES):
    one_hot = torch.zeros(1, N_FEATURES)
    one_hot[0, feat_idx] = 1.0
    with torch.no_grad():
        result = mlp_model.ae(one_hot)
        reconstructed = result[0] if isinstance(result, tuple) else result
        mse_loss = ((one_hot - reconstructed) ** 2).mean().item()
    one_hot_reconstructions.append(mse_loss)

one_hot_reconstructions = np.array(one_hot_reconstructions)

# Plot angular variance comparison
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Angular Variance by Feature (Zipf)",
        "Distribution Comparison",
        "AV vs Feature Probability",
        "Zipf vs Uniform MLP",
    ],
    specs=[[{}, {}], [{}, {}]],
)

features = list(range(N_FEATURES))

# Per-feature AV (Zipf models)
fig.add_trace(
    go.Scatter(
        x=features, y=relu_avs, name="TiedLinearRelu", mode="lines", opacity=0.7
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=features, y=mlp_avs, name="MLP (Zipf)", mode="lines"), row=1, col=1
)

# Box plot comparison
fig.add_trace(
    go.Box(y=relu_valid, name="TiedLinearRelu", boxpoints="outliers"), row=1, col=2
)
fig.add_trace(
    go.Box(y=mlp_valid, name="MLP (Zipf)", boxpoints="outliers"), row=1, col=2
)

# AV vs feature probability (scatter) - colored by reconstruction MSE (0 = good)
fig.add_trace(
    go.Scatter(
        x=ZIPF_PROBS,
        y=mlp_avs,
        mode="markers",
        name="MLP (Zipf)",
        marker=dict(
            size=5,
            opacity=0.8,
            color=one_hot_reconstructions,
            colorscale="RdYlGn_r",
            colorbar=dict(title="MSE Loss", x=0.46, y=0.23, len=0.4),
            showscale=True,
        ),
    ),
    row=2,
    col=1,
)

# Zipf vs Uniform MLP comparison
fig.add_trace(go.Box(y=mlp_valid, name="MLP Zipf", boxpoints="outliers"), row=2, col=2)

fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=2)
fig.update_xaxes(title_text="Feature p_active", type="log", row=2, col=1)
fig.update_yaxes(title_text="Angular Variance", row=2, col=1)
fig.update_yaxes(title_text="Angular Variance", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Angular Variance Analysis: Manifold Structure Detection",
    showlegend=True,
)
fig.show()

## 4. Jacobian PCA Analysis

For features with high angular variance, analyze the dimensionality of the direction manifold.

In [ ]:
# Find features with highest angular variance in MLP
valid_mask = ~np.isnan(mlp_avs)
sorted_indices = np.argsort(mlp_avs[valid_mask])[::-1]
valid_features = np.where(valid_mask)[0]
top_av_features = valid_features[sorted_indices[:10]]

print("Top 10 features by angular variance (MLP Zipf):")
for feat_idx in top_av_features:
    print(
        f"  Feature {feat_idx}: AV = {mlp_avs[feat_idx]:.6f}, p_active = {ZIPF_PROBS[feat_idx]:.4f}"
    )

In [ ]:
# PCA analysis for top features
def analyze_feature_pca(model, feat_idx, samples, n_samples=500):
    """Compute PCA eigenvalues for a feature's Jacobian directions."""
    active_mask = samples[:, feat_idx] > 0
    active_samples = samples[active_mask][:n_samples]

    jacs = compute_feature_jacobians(model, feat_idx, active_samples)
    eigenvalues, eigenvectors = jacobian_pca(jacs)

    return eigenvalues.detach().numpy(), eigenvectors


# Compute PCA for top features
pca_results = {}
for feat_idx in top_av_features[:5]:
    eigenvalues, _ = analyze_feature_pca(mlp_model, feat_idx, test_samples)
    pca_results[feat_idx] = eigenvalues

    # Compute explained variance ratio
    total_var = eigenvalues.sum()
    ratios = eigenvalues / total_var if total_var > 1e-10 else eigenvalues

    print(f"Feature {feat_idx} (AV={mlp_avs[feat_idx]:.4f}):")
    print(f"  Top 5 eigenvalue ratios: {ratios[:5].round(4)}")
    print(f"  Cumulative variance (5 components): {ratios[:5].sum():.4f}")

In [ ]:
# Plot PCA eigenvalue spectra
fig = go.Figure()

for feat_idx, eigenvalues in pca_results.items():
    total_var = eigenvalues.sum()
    ratios = eigenvalues / total_var if total_var > 1e-10 else eigenvalues

    fig.add_trace(
        go.Bar(
            x=list(range(1, len(eigenvalues) + 1)),
            y=ratios,
            name=f"Feature {feat_idx} (AV={mlp_avs[feat_idx]:.3f})",
            opacity=0.7,
        )
    )

fig.update_layout(
    title="PCA Eigenvalue Spectrum for High-AV Features",
    xaxis_title="Principal Component",
    yaxis_title="Explained Variance Ratio",
    barmode="group",
    height=400,
)
fig.show()

## 5. Direction vs Context Analysis

For high-AV features, identify which co-active features cause the largest direction rotations.

In [ ]:
# Analyze direction vs context for the highest-AV feature
top_feat = top_av_features[0]
print(f"Analyzing feature {top_feat} (highest AV = {mlp_avs[top_feat]:.6f})")

# Get active samples
active_mask = test_samples[:, top_feat] > 0
active_samples = test_samples[active_mask][:500]

# Compute Jacobians
jacs = compute_feature_jacobians(mlp_model, top_feat, active_samples)

# Direction vs context
context_result = direction_vs_context(
    mlp_model,
    feature_idx=top_feat,
    inputs=active_samples,
    jacobians=jacs,
)

print(f"\nMost influential co-active features:")
for feat_idx, magnitude in context_result["most_influential"]:
    print(
        f"  Feature {feat_idx}: correlation magnitude = {magnitude:.4f}, p_active = {ZIPF_PROBS[feat_idx]:.4f}"
    )

In [ ]:
# Visualize correlation magnitudes vs feature probability
corr_mags = context_result["correlation_magnitudes"].detach().numpy()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        f"Influence on Feature {top_feat}'s Direction",
        "Correlation Magnitude vs p_active",
    ],
)

# Bar chart
colors = ["red" if i == top_feat else "blue" for i in range(N_FEATURES)]
fig.add_trace(
    go.Bar(
        x=list(range(N_FEATURES)), y=corr_mags, marker_color=colors, showlegend=False
    ),
    row=1,
    col=1,
)

# Scatter: correlation vs probability
fig.add_trace(
    go.Scatter(
        x=ZIPF_PROBS,
        y=corr_mags,
        mode="markers",
        marker=dict(size=5, color=corr_mags, colorscale="Viridis"),
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="Correlation Magnitude", row=1, col=1)
fig.update_xaxes(title_text="Feature p_active", type="log", row=1, col=2)
fig.update_yaxes(title_text="Correlation Magnitude", row=1, col=2)

fig.update_layout(
    height=400, title_text=f"Context Dependence Analysis for Feature {top_feat}"
)
fig.show()

## 6. 3D Visualization (Low-Dimensional Sanity Check)

Train models with n_hidden=3 for direct visualization of Jacobian directions on a sphere.

In [ ]:
# Smaller models for 3D visualization
N_FEATURES_3D = 50
N_HIDDEN_3D = 3
N_EPOCHS_3D = 2000

ZIPF_PROBS_3D = [1 / (i + 1) for i in range(N_FEATURES_3D)]


def create_zipf_distribution_3d(seed=42):
    return SparseUniform(
        N_FEATURES_3D,
        ZIPF_PROBS_3D,
        generator=Generator().manual_seed(seed),
    )


print("Training 3D models...")

linear_3d = ToyModel(
    distribution=create_zipf_distribution_3d(),
    ae=TiedLinear(n_features=N_FEATURES_3D, n_hidden=N_HIDDEN_3D),
)
linear_3d.fit(n_epochs=N_EPOCHS_3D, batch_size=BATCH_SIZE)
print("Linear 3D trained.")

mlp_3d = ToyModel(
    distribution=create_zipf_distribution_3d(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES_3D,
        n_hidden=N_HIDDEN_3D,
        encoder_hidden_dim=N_HIDDEN_3D * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
)
mlp_3d.fit(n_epochs=N_EPOCHS_3D, batch_size=BATCH_SIZE)
print("MLP 3D trained.")

In [ ]:
# Generate test samples for 3D models
test_dist_3d = create_zipf_distribution_3d(seed=999)
test_samples_3d = test_dist_3d.sample(2000)

# Compute AV for all features in 3D models
print("Computing 3D angular variances...")
linear_3d_avs = []
mlp_3d_avs = []

for feat_idx in range(N_FEATURES_3D):
    active_mask = test_samples_3d[:, feat_idx] > 0
    active_samples = test_samples_3d[active_mask][:500]

    if len(active_samples) < 10:
        linear_3d_avs.append(np.nan)
        mlp_3d_avs.append(np.nan)
        continue

    linear_jacs = compute_feature_jacobians(linear_3d, feat_idx, active_samples)
    mlp_jacs = compute_feature_jacobians(mlp_3d, feat_idx, active_samples)

    linear_3d_avs.append(angular_variance(linear_jacs))
    mlp_3d_avs.append(angular_variance(mlp_jacs))

linear_3d_avs = np.array(linear_3d_avs)
mlp_3d_avs = np.array(mlp_3d_avs)

print(f"Linear 3D mean AV: {np.nanmean(linear_3d_avs):.6f}")
print(f"MLP 3D mean AV: {np.nanmean(mlp_3d_avs):.6f}")

In [ ]:
# Find highest-AV feature in 3D MLP
top_feat_3d = np.nanargmax(mlp_3d_avs)
print(f"Visualizing feature {top_feat_3d} (AV = {mlp_3d_avs[top_feat_3d]:.6f})")

test_samples_3d = test_dist_3d.sample(100000)


# Get samples and compute Jacobians
active_mask = test_samples_3d[:, top_feat_3d] > 0
active_samples_3d = test_samples_3d[active_mask][:500]

linear_jacs_3d = compute_feature_jacobians(linear_3d, top_feat_3d, active_samples_3d)
mlp_jacs_3d = compute_feature_jacobians(mlp_3d, top_feat_3d, active_samples_3d)

# Normalize to unit sphere
linear_normed = linear_jacs_3d / linear_jacs_3d.norm(dim=1, keepdim=True).clamp(
    min=1e-8
)
mlp_normed = mlp_jacs_3d / mlp_jacs_3d.norm(dim=1, keepdim=True).clamp(min=1e-8)

# 3D scatter plot
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=["TiedLinear (should cluster)", "MLPAutoencoder (may spread)"],
)

# Linear model
fig.add_trace(
    go.Scatter3d(
        x=linear_normed[:, 0].detach().cpu().numpy(),
        y=linear_normed[:, 1].detach().cpu().numpy(),
        z=linear_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(size=3, color="blue", opacity=0.6),
        name="Linear",
    ),
    row=1,
    col=1,
)

# MLP model - color by most common co-active feature
# Find most influential feature for this one
ctx = direction_vs_context(mlp_3d, top_feat_3d, active_samples_3d, mlp_jacs_3d)
influential_feat = ctx["most_influential"][0][0] if ctx["most_influential"] else 1
colors = active_samples_3d[:, influential_feat].numpy()

fig.add_trace(
    go.Scatter3d(
        x=mlp_normed[:, 0].detach().cpu().numpy(),
        y=mlp_normed[:, 1].detach().cpu().numpy(),
        z=mlp_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(
            size=3,
            color=colors,
            colorscale="Viridis",
            colorbar=dict(title=f"Feature {influential_feat}", x=1.0),
            opacity=0.6,
        ),
        name="MLP",
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=f"Jacobian Directions for Feature {top_feat_3d} on Unit Sphere (Zipf Distribution)",
    height=500,
    showlegend=False,
)
fig.show()

In [ ]:
# Plot Jacobian vectors for top N features - sorted by descending angular variance
import plotly.express as px
import numpy as np

# ============ CONFIGURATION ============
TOP_N_FEATURES = 30  # Change this to show top 1, 5, 10, or any number of features
# Set to None to show all features
# =======================================

test_samples_3d = test_dist_3d.sample(1000000)

# Scale factor for visibility
scale = 2.0

fig = go.Figure()

# Sort features by descending angular variance
valid_avs = [(i, av) for i, av in enumerate(mlp_3d_avs) if not np.isnan(av) and av > 0]
valid_avs.sort(key=lambda x: x[1], reverse=True)

# Select top N features
if TOP_N_FEATURES is not None:
    features_to_plot = [idx for idx, _ in valid_avs[:TOP_N_FEATURES]]
else:
    features_to_plot = [idx for idx, _ in valid_avs]

print(f"Plotting top {len(features_to_plot)} features by angular variance:")
for rank, (idx, av) in enumerate(valid_avs[: len(features_to_plot)], 1):
    print(f"  {rank}. Feature {idx}: AV = {av:.6f}")

# Use log scale for better color spread
av_valid = mlp_3d_avs[~np.isnan(mlp_3d_avs)]
av_log_min, av_log_max = np.log10(av_valid.min()), np.log10(av_valid.max())


def av_to_color(av):
    """Map angular variance to Viridis color using log scale."""
    if np.isnan(av) or av <= 0:
        return "rgb(128, 128, 128)"
    log_av = np.log10(av)
    norm_av = (log_av - av_log_min) / (av_log_max - av_log_min)
    norm_av = np.clip(norm_av, 0, 1)
    return px.colors.sample_colorscale("Viridis", [norm_av])[0]


# Collect all one-hot embeddings for selected features
all_one_hot_pts = []

for feat_idx in features_to_plot:
    active_mask = test_samples_3d[:, feat_idx] > 0
    n_active = active_mask.sum().item()

    if n_active < 10:
        continue

    active_samples = test_samples_3d[active_mask][:100]
    jacs = compute_feature_jacobians(mlp_3d, feat_idx, active_samples)

    with torch.no_grad():
        embedded = mlp_3d.ae.encode(active_samples)
        one_hot = torch.zeros(1, N_FEATURES_3D)
        one_hot[0, feat_idx] = 1.0
        one_hot_embedded = mlp_3d.ae.encode(one_hot)

    jac_vectors = jacs.detach().cpu().numpy()
    embed_pts = embedded.detach().cpu().numpy()
    one_hot_pt = one_hot_embedded.detach().cpu().numpy()[0]
    all_one_hot_pts.append((feat_idx, one_hot_pt, mlp_3d_avs[feat_idx]))

    end_pts = embed_pts + scale * jac_vectors

    # Build line segments
    xs, ys, zs = [], [], []
    for i in range(len(embed_pts)):
        xs.extend([embed_pts[i, 0], end_pts[i, 0], None])
        ys.extend([embed_pts[i, 1], end_pts[i, 1], None])
        zs.extend([embed_pts[i, 2], end_pts[i, 2], None])

    feat_color = av_to_color(mlp_3d_avs[feat_idx])

    # Lines
    fig.add_trace(
        go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="lines",
            line=dict(color=feat_color, width=2),
            name=f"Feature {feat_idx} (AV={mlp_3d_avs[feat_idx]:.4f})",
            opacity=0.3,
            legendgroup=f"feat_{feat_idx}",
            showlegend=True,
        )
    )

    # Arrowheads as cones at endpoints
    # Compute direction vectors (normalized)
    directions = jac_vectors / (
        np.linalg.norm(jac_vectors, axis=1, keepdims=True) + 1e-8
    )

    fig.add_trace(
        go.Cone(
            x=end_pts[:, 0],
            y=end_pts[:, 1],
            z=end_pts[:, 2],
            u=directions[:, 0],
            v=directions[:, 1],
            w=directions[:, 2],
            sizemode="absolute",
            sizeref=0.8,
            colorscale=[[0, feat_color], [1, feat_color]],
            showscale=False,
            opacity=0.5,
            legendgroup=f"feat_{feat_idx}",
            showlegend=False,
        )
    )

# Compute origin correctly
with torch.no_grad():
    zero_input = torch.zeros(1, N_FEATURES_3D)
    origin = mlp_3d.ae.encode(zero_input).detach().cpu().numpy()[0]

# Add origin
fig.add_trace(
    go.Scatter3d(
        x=[origin[0]],
        y=[origin[1]],
        z=[origin[2]],
        mode="markers",
        marker=dict(size=8, color="black"),
        name="Origin",
    )
)

# Add unit sphere centered at origin
u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
x_sphere = origin[0] + np.outer(np.cos(u), np.sin(v))
y_sphere = origin[1] + np.outer(np.sin(u), np.sin(v))
z_sphere = origin[2] + np.outer(np.ones(np.size(u)), np.cos(v))

fig.add_trace(
    go.Surface(
        x=x_sphere,
        y=y_sphere,
        z=z_sphere,
        opacity=0.15,
        colorscale=[[0, "lightgray"], [1, "lightgray"]],
        showscale=False,
        name="Unit Sphere",
        hoverinfo="skip",
    )
)

# One-hot embeddings with log-scaled colors
one_hot_x = [pt[0] for _, pt, _ in all_one_hot_pts]
one_hot_y = [pt[1] for _, pt, _ in all_one_hot_pts]
one_hot_z = [pt[2] for _, pt, _ in all_one_hot_pts]
one_hot_avs = [av for _, _, av in all_one_hot_pts]
one_hot_log_avs = [np.log10(av) if av > 0 else av_log_min for av in one_hot_avs]
one_hot_labels = [f"Feature {idx} (AV={av:.4f})" for idx, _, av in all_one_hot_pts]

fig.add_trace(
    go.Scatter3d(
        x=one_hot_x,
        y=one_hot_y,
        z=one_hot_z,
        mode="markers",
        marker=dict(
            size=6,
            color=one_hot_log_avs,
            colorscale="Viridis",
            colorbar=dict(
                title="log₁₀(AV)",
                tickvals=[av_log_min, (av_log_min + av_log_max) / 2, av_log_max],
                ticktext=[
                    f"{10**av_log_min:.3f}",
                    f"{10 ** ((av_log_min + av_log_max) / 2):.3f}",
                    f"{10**av_log_max:.3f}",
                ],
            ),
            symbol="diamond",
        ),
        text=one_hot_labels,
        name="One-hot embeddings",
    )
)

title_text = f"Jacobian Vectors for Top {len(features_to_plot)} Features by Angular Variance (log scale, scale={scale}x)"
if TOP_N_FEATURES is None:
    title_text = f"Jacobian Vectors for All {N_FEATURES_3D} Features (colored by Angular Variance, log scale, scale={scale}x)"

fig.update_layout(
    title=title_text,
    scene=dict(
        xaxis_title="Hidden dim 1",
        yaxis_title="Hidden dim 2",
        zaxis_title="Hidden dim 3",
    ),
    height=700,
    showlegend=True,
)
fig.show()

## 7. Experiment C: Reconstruction Quality Comparison

Compare final reconstruction loss. If MLP achieves significantly lower loss,
the nonlinear encoding is doing useful work.

In [ ]:
# Compute reconstruction loss on held-out data
test_dist_eval = create_zipf_distribution(seed=12345)
eval_samples = test_dist_eval.sample(5000)


def compute_reconstruction_loss(model, samples):
    with torch.no_grad():
        result = model.ae(samples)
        # Handle case where ae returns a tuple (reconstructed, encoded) or just a tensor
        reconstructed = result[0] if isinstance(result, tuple) else result
        mse = ((samples - reconstructed) ** 2).mean().item()
    return mse


# linear_loss = compute_reconstruction_loss(linear_model, eval_samples)
relu_loss = compute_reconstruction_loss(relu_model, eval_samples)
mlp_loss = compute_reconstruction_loss(mlp_model, eval_samples)

# Uniform comparison
eval_samples_uniform = create_uniform_distribution(seed=12345).sample(5000)
# mlp_uniform_loss = compute_reconstruction_loss(mlp_uniform_model, eval_samples_uniform)

print("=" * 60)
print("RECONSTRUCTION LOSS COMPARISON (held-out data)")
print("=" * 60)
# print(f"TiedLinear (Zipf):      {linear_loss:.6f}")
print(f"TiedLinearRelu (Zipf):  {relu_loss:.6f}")
print(f"MLPAutoencoder (Zipf):  {mlp_loss:.6f}")
# print(f"MLPAutoencoder (Unif):  {mlp_uniform_loss:.6f}")
print()
# print(
#     f"MLP vs Linear improvement (Zipf): {(linear_loss - mlp_loss) / linear_loss * 100:.2f}%"
# )

## 8. Summary

Key findings from this experiment:

In [ ]:
print("=" * 70)
print("EXPERIMENT 3.2 SUMMARY: Manifold Structure with Zipf Distribution")
print("=" * 70)
print()
print("1. ANGULAR VARIANCE (manifold structure detection):")
print(f"   - Linear baseline: {np.nanmean(linear_avs):.6f} (expected ~0)")
print(f"   - MLP (Zipf):      {np.nanmean(mlp_avs):.6f}")
print(f"   - MLP (Uniform):   {np.nanmean(mlp_uniform_avs):.6f}")
print()

av_increase = np.nanmean(mlp_avs) / max(np.nanmean(linear_avs), 1e-10)
if av_increase > 10:
    print("   => SIGNIFICANT manifold structure detected in MLP!")
elif av_increase > 2:
    print("   => Moderate manifold structure detected.")
else:
    print("   => MLP converged to near-linear encoding.")

print()
print("2. RECONSTRUCTION QUALITY:")
print(f"   - Linear: {linear_loss:.6f}")
print(f"   - MLP:    {mlp_loss:.6f}")
improvement = (linear_loss - mlp_loss) / linear_loss * 100
print(f"   - Improvement: {improvement:.2f}%")

if improvement > 10:
    print("   => MLP's nonlinearity provides substantial benefit!")
elif improvement > 2:
    print("   => MLP provides modest improvement.")
else:
    print("   => Linearity is near-optimal for this task.")

print()
print("3. ZIPF vs UNIFORM:")
zipf_av = np.nanmean(mlp_avs)
uniform_av = np.nanmean(mlp_uniform_avs)
print(f"   - Zipf AV:    {zipf_av:.6f}")
print(f"   - Uniform AV: {uniform_av:.6f}")
if zipf_av > uniform_av * 1.2:
    print("   => Zipf distribution promotes more manifold structure!")
elif uniform_av > zipf_av * 1.2:
    print("   => Uniform distribution promotes more manifold structure.")
else:
    print("   => Similar manifold structure across distributions.")